# The Math Behind Machine Learning, Visualized
### From dot products to a generative screensaver

This notebook is a guided tour of the mathematical ideas that power modern ML —
vectors, matrices, nonlinearity, gradients, and learned functions — built up
through small interactive experiments. We end by training a tiny neural network
to define a *flow field*, then turning that field into an evolving particle
screensaver designed for a **30" × 10" vertical display** (1:3 portrait).

Dependencies: `numpy`, `matplotlib`. No PyTorch, no TensorFlow — every gradient
is computed by hand so the math stays visible. Everything below is roughly 150
lines of real code; the rest is explanation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML

np.random.seed(7)

# Inline animations can be ~30–60 MB once we go long — bump the embed cap.
plt.rcParams['animation.embed_limit'] = 200   # MB

# Dark theme so plots feel like the screensaver they're building toward
for k, v in {
    'figure.facecolor': '#0b0b14', 'axes.facecolor': '#0b0b14',
    'axes.edgecolor': '#444',     'axes.labelcolor': '#ddd',
    'xtick.color': '#888',         'ytick.color': '#888',
    'text.color': '#ddd',          'axes.titlecolor': '#eee',
}.items():
    plt.rcParams[k] = v

## 1. Vectors are ideas

Every input to a neural network — a pixel patch, a token, an audio frame — becomes
a vector of numbers. The geometry of these vectors is the geometry of meaning.
The **dot product** between two vectors measures how aligned they are:

$$\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

+1 means "same idea," 0 means "unrelated," -1 means "opposite."

In [ ]:
def cos_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

fig, ax = plt.subplots(figsize=(5, 5))
a = np.array([1.0, 0.0])
for t in np.linspace(0, 2 * np.pi, 28, endpoint=False):
    b = np.array([np.cos(t), np.sin(t)])
    s = cos_sim(a, b)
    ax.arrow(0, 0, b[0], b[1], head_width=0.04,
             color=plt.cm.coolwarm(0.5 * (s + 1)), alpha=0.95)
ax.arrow(0, 0, a[0], a[1], head_width=0.07, color='white', linewidth=2)
ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.25); ax.set_aspect('equal')
ax.set_title('Cosine similarity to the white vector\n(red = aligned, blue = opposite)')
plt.show()

## 2. Matrices bend space

A matrix is a *function* — multiplication by a $2\times2$ matrix takes every point
in the plane and moves it somewhere new. Three families of motion together
describe every linear map: **scale**, **rotate**, **shear**.

Watch what each does to a uniform grid of points.

In [ ]:
def show_transform(M, ax, title):
    grid = np.mgrid[-1:1.01:0.1, -1:1.01:0.1].reshape(2, -1)
    out = M @ grid
    ax.scatter(grid[0], grid[1], s=2, c='#444')
    ax.scatter(out[0], out[1], s=4, c='#ffb84a')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
show_transform(np.array([[1.5, 0], [0, 0.6]]), axes[0], 'scale')
show_transform(np.array([[np.cos(0.7), -np.sin(0.7)],
                         [np.sin(0.7),  np.cos(0.7)]]), axes[1], 'rotate')
show_transform(np.array([[1, 0.7], [0.0, 1]]), axes[2], 'shear')
plt.show()

## 3. Nonlinearity unlocks expressivity

Stack two linear transforms and you get… another linear transform. To curve
decision boundaries — to fit anything more complicated than a line — you need a
*nonlinearity* between layers. The classics are `tanh`, `relu`, `gelu`.

Watch how `tanh` smashes a stretched grid into a curved cushion. That bending is
what makes deep networks expressive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
grid = np.mgrid[-2:2.01:0.1, -2:2.01:0.1].reshape(2, -1)
W = np.array([[1.2, 0.5], [-0.4, 1.1]])
linear = W @ grid
nonlinear = np.tanh(W @ grid * 1.5) * 1.5
axes[0].scatter(linear[0], linear[1], s=2, c='#88c'); axes[0].set_title('linear:  Wx')
axes[1].scatter(nonlinear[0], nonlinear[1], s=2, c='#fc8'); axes[1].set_title('nonlinear:  tanh(Wx)')
for a in axes:
    a.set_aspect('equal'); a.set_xlim(-3, 3); a.set_ylim(-3, 3)
plt.show()

## 4. Gradients — the compass of learning

A neural network learns by minimizing a loss function. The **gradient** of the
loss points in the direction of fastest *increase*, so we step in the opposite
direction:

$$\theta_{t+1} = \theta_t - \eta\,\nabla_\theta \mathcal{L}(\theta_t)$$

That single update rule is the engine of essentially all modern ML.

Below: a wavy loss surface, with descent paths started from five random points.
Each path follows the gradient downhill until it settles in a basin.

In [ ]:
def loss(x, y):
    return np.sin(x) * np.cos(y) + 0.1 * (x**2 + y**2)

def grad(x, y):
    return (np.cos(x) * np.cos(y) + 0.2 * x,
            -np.sin(x) * np.sin(y) + 0.2 * y)

xs = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(xs, xs)
Z = loss(X, Y)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(X, Y, Z, 30, cmap='magma')
for sx, sy in [(-2.5, 2.5), (2.5, 2.5), (-2.5, -2.5), (2.5, -2.5), (0.4, 1.5)]:
    p = [(sx, sy)]
    for _ in range(120):
        gx, gy = grad(*p[-1])
        p.append((p[-1][0] - 0.08 * gx, p[-1][1] - 0.08 * gy))
    p = np.array(p)
    ax.plot(p[:, 0], p[:, 1], '-', color='cyan', alpha=0.85, linewidth=1.2)
    ax.plot(p[0, 0], p[0, 1], 'o', color='white', markersize=4)
ax.set_aspect('equal')
ax.set_title('Gradient descent paths on a wavy loss surface')
plt.show()

## 5. A tiny neural network, by hand

Time to put it all together. We'll build a 2-input, 1-output MLP with one hidden
layer of `tanh` units, and train it with simple gradient descent to fit a
scalar field $\psi(x, y)$ — a *potential* over a tall 1:3 region of the plane.

$$\psi(\mathbf{x}) = W_2 \,\tanh(W_1\mathbf{x} + b_1) + b_2$$

Backprop is short enough to write out directly. No autograd.

In [ ]:
def init_mlp(in_dim=2, hidden=64, out_dim=1, scale=0.8):
    return {
        'W1': np.random.randn(hidden, in_dim) * scale,
        'b1': np.zeros(hidden),
        'W2': np.random.randn(out_dim, hidden) * scale,
        'b2': np.zeros(out_dim),
    }

def forward(p, X):
    h = np.tanh(X @ p['W1'].T + p['b1'])    # (N, H)
    y = h @ p['W2'].T + p['b2']             # (N, 1)
    return y, h

def train_step(p, X, y, lr=3e-2):
    yhat, h = forward(p, X)
    err = yhat - y
    N = X.shape[0]
    gW2 = err.T @ h / N
    gb2 = err.mean(0)
    dh  = err @ p['W2'] * (1 - h**2)        # backprop through tanh
    gW1 = dh.T @ X / N
    gb1 = dh.mean(0)
    p['W1'] -= lr * gW1; p['b1'] -= lr * gb1
    p['W2'] -= lr * gW2; p['b2'] -= lr * gb2
    return float((err**2).mean())

In [ ]:
# Target field: a stack of Gaussian bumps and dips arranged vertically,
# matching the 1:3 aspect ratio of the eventual screensaver canvas.
DOM_W, DOM_H = 1.6, 4.8                     # half-extents in x and y

def target_field(X):
    centers = np.array([
        [-1.0, -4.0], [ 0.7, -3.0], [-0.6, -1.5],
        [ 0.5,  0.0], [-0.8,  1.5], [ 0.6,  3.0], [-0.3,  4.2],
    ])
    weights = np.array([1.0, -1.0, 0.9, -0.8, 1.1, -0.7, 0.8])
    z = np.zeros(len(X))
    for c, w in zip(centers, weights):
        d2 = ((X - c)**2).sum(axis=1)
        z += w * np.exp(-d2 / 0.9)
    return z

N = 4000
X_train = np.column_stack([
    np.random.uniform(-DOM_W, DOM_W, size=N),
    np.random.uniform(-DOM_H, DOM_H, size=N),
])
y_train = target_field(X_train)[:, None]

mlp = init_mlp(hidden=64)
losses = []
for step in range(5000):
    idx = np.random.choice(N, size=256, replace=False)
    losses.append(train_step(mlp, X_train[idx], y_train[idx], lr=3e-2))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses, color='#ffb84a', linewidth=0.7)
ax.set_yscale('log'); ax.set_xlabel('step'); ax.set_ylabel('MSE')
ax.set_title('Training loss (log scale)')
plt.show()

In [ ]:
gx = np.linspace(-DOM_W, DOM_W, 80)
gy = np.linspace(-DOM_H, DOM_H, 240)
GX, GY = np.meshgrid(gx, gy)
pts = np.stack([GX.ravel(), GY.ravel()], axis=1)
target_grid  = target_field(pts).reshape(GX.shape)
learned_grid = forward(mlp, pts)[0].reshape(GX.shape)

fig, axes = plt.subplots(1, 2, figsize=(6, 9))
for ax, img, title in zip(axes,
                          [target_grid, learned_grid],
                          ['target potential', 'MLP-learned potential']):
    ax.imshow(img, extent=(-DOM_W, DOM_W, -DOM_H, DOM_H),
              origin='lower', cmap='magma')
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')
plt.show()

## 6. From a learned potential to a flow field

The gradient $\nabla\psi$ of our scalar field points uphill. The *perpendicular*
gradient $(\partial_y\psi,\,-\partial_x\psi)$ traces along level sets — a
divergence-free flow that swirls around the bumps and dips. In fluid dynamics
this is called a **stream function**, and physics-informed networks use this
trick to guarantee mass-conserving flows.

We can compute $\nabla\psi$ analytically by chain-ruling through the network —
no autograd needed, since it's just one tanh layer.

In [ ]:
def potential_and_grad(p, X):
    """ψ(X) and ∇ψ(X) via analytic backprop through the MLP."""
    pre = X @ p['W1'].T + p['b1']           # (N, H)
    h = np.tanh(pre)
    psi = h @ p['W2'].T + p['b2']           # (N, 1)
    grad = ((1 - h**2) * p['W2']) @ p['W1'] # (N, 2)
    return psi[:, 0], grad

def stream_flow(p, X, swirl=1.0, drift=(0.0, 0.0)):
    """Velocity = perpendicular gradient (divergence-free) + constant drift."""
    _, g = potential_and_grad(p, X)
    flow = np.stack([g[:, 1], -g[:, 0]], axis=1) * swirl
    flow[:, 0] += drift[0]; flow[:, 1] += drift[1]
    return flow

In [ ]:
gx = np.linspace(-DOM_W, DOM_W, 18)
gy = np.linspace(-DOM_H, DOM_H, 54)
GX, GY = np.meshgrid(gx, gy)
pts = np.stack([GX.ravel(), GY.ravel()], axis=1)
flow = stream_flow(mlp, pts, swirl=2.0)
U = flow[:, 0].reshape(GX.shape); V = flow[:, 1].reshape(GX.shape)

fig, ax = plt.subplots(figsize=(4, 11))
ax.imshow(learned_grid, extent=(-DOM_W, DOM_W, -DOM_H, DOM_H),
          origin='lower', cmap='magma', alpha=0.55)
ax.streamplot(GX, GY, U, V, color='cyan', linewidth=0.6, density=1.2, arrowsize=0.6)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Flow field from the learned stream function', color='#eee')
plt.show()

## 7. The artifact — a vertical neural flow screensaver

Now we drop thousands of particles into the flow, give them faint glowing
trails, and animate. The canvas is **1:3 portrait**, sized for a 30" × 10"
vertical display.

Two design choices that make it feel alive:

1. A small **upward drift** is added to the flow so particles rise like embers,
   while still being shaped by the swirls of the learned field. New particles
   spawn at the bottom to keep the field populated.
2. The MLP's weights are slowly perturbed each frame, so the field **never
   exactly repeats** — the screensaver is technically a non-stationary stochastic
   process.

Trails come from a separate trick: instead of plotting markers, we splat
particles into a 2D image buffer, fade the buffer slightly each frame, and
display it through a custom palette. That gives the dense, glowing look that
scatterplots can't.

In [ ]:
class TrailCanvas:
    """An accumulating intensity buffer that fades each frame."""
    def __init__(self, w_px, h_px, fade=0.93, palette=None):
        self.w, self.h = w_px, h_px
        self.buf = np.zeros((h_px, w_px), dtype=np.float32)
        self.fade = fade
        self.palette = palette

    def splat(self, P, intensity, dom_w, dom_h):
        self.buf *= self.fade
        px = ((P[:, 0] / (2 * dom_w) + 0.5) * (self.w - 1)).astype(np.int32)
        py = ((P[:, 1] / (2 * dom_h) + 0.5) * (self.h - 1)).astype(np.int32)
        m = (px >= 0) & (px < self.w) & (py >= 0) & (py < self.h)
        np.add.at(self.buf, (py[m], px[m]), intensity[m])

    def render(self, gamma=0.7, scale=1.5):
        return self.palette(np.clip(self.buf * scale, 0, 1) ** gamma)


class NeuralFlow:
    """Particle simulator over the MLP's stream function."""
    def __init__(self, mlp, n=3000, dom_w=DOM_W, dom_h=DOM_H,
                 swirl=2.0, drift=(0.0, 0.6),
                 weight_drift=8e-5, seed=0):
        self.mlp = {k: v.copy() for k, v in mlp.items()}
        self.dom_w, self.dom_h = dom_w, dom_h
        self.swirl = swirl
        self.drift = np.array(drift, dtype=float)
        self.weight_drift = weight_drift
        self.rng = np.random.default_rng(seed)
        self.P = self._spawn(n)
        self.age = self.rng.uniform(0, 1, size=n)
        self.life = self.rng.uniform(180, 400, size=n)

    def _spawn(self, n):
        x = self.rng.uniform(-self.dom_w, self.dom_w, size=n)
        y = self.rng.uniform(-self.dom_h, self.dom_h, size=n)
        return np.stack([x, y], axis=1)

    def _respawn(self, mask):
        n = int(mask.sum())
        if n == 0:
            return
        x = self.rng.uniform(-self.dom_w, self.dom_w, size=n)
        y = self.rng.uniform(-self.dom_h, -self.dom_h + 0.2 * 2 * self.dom_h, size=n)
        self.P[mask] = np.stack([x, y], axis=1)
        self.age[mask] = 0
        self.life[mask] = self.rng.uniform(180, 400, size=n)

    def step(self, dt=0.05):
        flow = stream_flow(self.mlp, self.P, swirl=self.swirl, drift=tuple(self.drift))
        # cap step size so fast regions don't tear the trail
        speed = np.linalg.norm(flow, axis=1, keepdims=True) + 1e-6
        flow = flow / np.maximum(speed / 3.0, 1.0)
        self.P = self.P + flow * dt
        self.age += 1
        out = ((self.P[:, 0] < -self.dom_w) | (self.P[:, 0] >  self.dom_w) |
               (self.P[:, 1] >  self.dom_h) | (self.age > self.life))
        self._respawn(out)
        for k in ('W1', 'W2'):
            self.mlp[k] += self.rng.standard_normal(self.mlp[k].shape) * self.weight_drift
        return flow


# Custom palette: deep purple -> magenta -> warm gold ('embers')
embers = LinearSegmentedColormap.from_list(
    'embers', ['#0a0014', '#240050', '#7a1a70', '#e64d3d', '#ffd166'])

In [ ]:
# Live preview animation — 720 frames at 22fps (~33s loop).
# dt is the per-frame simulation step; lowering it slows particle motion.
canvas = TrailCanvas(w_px=200, h_px=600, fade=0.96, palette=embers)
sim    = NeuralFlow(mlp, n=4000, swirl=2.2, drift=(0.0, 0.35),
                    weight_drift=5e-5, seed=11)

fig = plt.figure(figsize=(3.33, 10), dpi=90)
fig.patch.set_facecolor('#000')
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor('#000'); ax.axis('off')
img = ax.imshow(canvas.render(), origin='lower', interpolation='bilinear', aspect='auto')

def update(frame):
    flow = sim.step(dt=0.03)
    speed = np.linalg.norm(flow, axis=1)
    intensity = 0.4 + np.minimum(speed, 3.0) * 0.25
    canvas.splat(sim.P, intensity, sim.dom_w, sim.dom_h)
    img.set_data(canvas.render(gamma=0.7, scale=1.4))
    return (img,)

anim = animation.FuncAnimation(fig, update, frames=720, interval=45, blit=True)
plt.close(fig)
HTML(anim.to_jshtml(default_mode='loop'))

In [ ]:
# Save the looping preview. MP4 is ~10× smaller than GIF if you have ffmpeg.
import shutil
if shutil.which('ffmpeg'):
    anim.save('neural_flow_screensaver.mp4',
              writer=animation.FFMpegWriter(fps=22, bitrate=2000, codec='libx264',
                                            extra_args=['-pix_fmt', 'yuv420p']))
    print('Saved → neural_flow_screensaver.mp4')
else:
    anim.save('neural_flow_screensaver.gif', writer=animation.PillowWriter(fps=22))
    print('Saved → neural_flow_screensaver.gif')

In [ ]:
# A high-resolution still — your wallpaper poster. Renders by simulating
# silently for ~600 frames, accumulating into a larger trail buffer.
hi_canvas = TrailCanvas(w_px=540, h_px=1620, fade=0.94, palette=embers)
hi_sim    = NeuralFlow(mlp, n=8000, swirl=2.0, drift=(0.0, 0.55),
                       weight_drift=5e-5, seed=42)
for _ in range(600):
    flow = hi_sim.step(dt=0.04)
    intensity = 0.35 + np.minimum(np.linalg.norm(flow, axis=1), 3.0) * 0.22
    hi_canvas.splat(hi_sim.P, intensity, hi_sim.dom_w, hi_sim.dom_h)

fig, ax = plt.subplots(figsize=(3.33, 10), dpi=200)
fig.patch.set_facecolor('#000'); ax.axis('off')
ax.imshow(hi_canvas.render(gamma=0.7, scale=1.5), origin='lower', aspect='auto')
plt.savefig('neural_flow_still.png', facecolor='#000', dpi=200,
            bbox_inches='tight', pad_inches=0)
plt.show()
print('Saved → neural_flow_still.png')

## Where to take it next

- **Different potentials.** Fit the MLP to a single grayscale image (treat pixel
  intensity as $\psi$). The flow will trace your photo's contours.
- **Time as input.** Make the network 3-in / 1-out and feed it `(x, y, t)`. The
  field morphs continuously instead of via random weight drift.
- **Audio reactive.** Modulate `swirl`, `drift`, or `weight_drift` from an audio
  envelope. The screensaver dances to music.
- **Physics-informed losses.** Add a term so the learned $\psi$ satisfies
  Laplace's equation $\nabla^2 \psi = 0$. The animation gains the look of real
  potential flow.
- **Wraparound topology.** Map the canvas to a torus or Möbius strip — particles
  that exit one side reappear on the other, and the geometry rewrites the field.
- **Bigger nets, learned in real time.** Stream training data and watch the
  field reorganize as it learns. The screensaver becomes a live window into
  optimization itself.

The whole thing is roughly 150 lines of NumPy. Bend it however you like.

## 8. Level up — a GPU-accelerated CPPN screensaver

The CPU version is satisfying but limited: 4000 particles, one tanh layer, a
field that morphs only via random weight drift. Now we move everything to the
GPU and trade up to a more interesting object: a **time-conditioned
Compositional Pattern-Producing Network** (CPPN).

A CPPN is a small MLP whose inputs are coordinates `(x, y, t)` and whose
outputs encode an image or, here, a field. They're a research lineage that
goes back to NEAT (Stanley, 2007) and runs through neural cellular automata
and modern generative art. Two things make them produce beautiful patterns
without any training:

1. **Mixed activations** per layer — we cycle through `sin`, `tanh`, `gauss`.
   Each one carves space differently, so even random weights compose into
   organic structure.
2. **High-magnitude weight init** (SIREN-style for the `sin` layers) — pushes
   activations through their full nonlinear range, creating fine detail
   instead of mushy gradients.

The network outputs **4 values per point**: one stream function $\psi$ for the
flow, and three logits that become per-particle RGB color via `sigmoid`. So
particles drift through one shared flow field but pick up color *from where
they are in the network's output space*. Color isn't a fixed palette — it's
the network's own representation, visualized.

Time is the third input. We feed `t = 0.5·sin(2π·frame/period)` so the field
morphs smoothly and the animation loops. No retraining, no weight drift — just
the network exploring its own time-slice.

The whole forward + backward pass for ~90,000 particles runs in ~25 ms on a
GTX 1070. The bottleneck is `autograd.grad` for $\nabla\psi$; we use
`is_grads_batched=True` so the backward pass handles the whole batch at once.

In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB')


class CPPN(nn.Module):
    """(x, y, t) → (ψ, R, G, B logits).  No training — random init is the art."""
    def __init__(self, hidden=96, n_layers=4, seed=42, w_scale=8.0):
        super().__init__()
        torch.manual_seed(seed)
        self.layers = nn.ModuleList()
        in_d = 3
        for _ in range(n_layers):
            self.layers.append(nn.Linear(in_d, hidden))
            in_d = hidden
        self.out = nn.Linear(in_d, 4)
        with torch.no_grad():
            for lin in self.layers:
                b = w_scale / lin.in_features
                lin.weight.uniform_(-b, b)
                lin.bias.uniform_(-0.3, 0.3)
            self.out.weight.uniform_(-0.6, 0.6)
            self.out.bias.zero_()

    def forward(self, x):
        # cycle through three activations — each carves space differently
        acts = [torch.sin, torch.tanh, lambda z: torch.exp(-z * z), torch.sin]
        for i, lin in enumerate(self.layers):
            x = acts[i % len(acts)](lin(x))
        return self.out(x)


def gpu_stream_flow(model, P, t_val, swirl=1.5, drift=(0.0, 0.2)):
    """Flow + per-particle color from one CPPN forward + one backward.
    Color uses *centered* logits so each particle gets a saturated hue
    instead of an additively-washed gray."""
    P_g = P.detach().requires_grad_(True)
    t_col = torch.full((P_g.shape[0], 1), float(t_val), device=P_g.device)
    xyt = torch.cat([P_g, t_col], dim=1)
    out = model(xyt)                          # (N, 4)
    psi, color_logits = out[:, 0], out[:, 1:]
    g = torch.autograd.grad(psi.sum(), P_g)[0]   # (N, 2)
    flow = torch.stack([g[:, 1], -g[:, 0]], dim=1) * swirl
    flow[:, 0] += drift[0]; flow[:, 1] += drift[1]
    centered = color_logits - color_logits.mean(dim=1, keepdim=True)
    rgb = torch.sigmoid(centered * 3.0)       # (N, 3) — channel-distinct color
    return flow.detach(), rgb.detach()

In [ ]:
class GPUTrailCanvas:
    """RGB trail buffer on the GPU.  Each particle splats its own color."""
    def __init__(self, w, h, fade=0.95, device=device):
        self.w, self.h = w, h
        self.buf = torch.zeros(3, h, w, device=device)
        self.fade = fade

    def splat(self, P, intensity, rgb, dom_w, dom_h):
        self.buf.mul_(self.fade)
        px = ((P[:, 0] / (2 * dom_w) + 0.5) * (self.w - 1)).long().clamp_(0, self.w - 1)
        py = ((P[:, 1] / (2 * dom_h) + 0.5) * (self.h - 1)).long().clamp_(0, self.h - 1)
        idx = py * self.w + px
        for c in range(3):
            self.buf[c].view(-1).index_add_(0, idx, intensity * rgb[:, c])

    def render(self, gamma=0.7, scale=1.0):
        out = torch.clamp(self.buf * scale, 0, 1) ** gamma
        return out.permute(1, 2, 0).cpu().numpy()


class GPUNeuralFlow:
    """Particle simulator over a CPPN stream function, all on GPU."""
    def __init__(self, model, n=90000, dom_w=1.5, dom_h=4.5,
                 swirl=1.5, drift=(0.0, 0.2), seed=0):
        self.model = model.to(device).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.dom_w, self.dom_h = dom_w, dom_h
        self.swirl = swirl; self.drift = drift
        self.n = n
        torch.manual_seed(seed)
        self.P = self._spawn(n)
        self.age = torch.rand(n, device=device) * 300
        self.life = torch.rand(n, device=device) * 220 + 200

    def _spawn(self, n):
        x = (torch.rand(n, device=device) * 2 - 1) * self.dom_w
        y = (torch.rand(n, device=device) * 2 - 1) * self.dom_h
        return torch.stack([x, y], dim=1)

    def _respawn(self, mask):
        n = int(mask.sum().item())
        if n == 0: return
        nx = (torch.rand(n, device=device) * 2 - 1) * self.dom_w
        ny = (torch.rand(n, device=device) * 0.4 - 1.0) * self.dom_h
        self.P = self.P.clone()
        self.P[mask] = torch.stack([nx, ny], dim=1)
        self.age[mask] = 0
        self.life[mask] = torch.rand(n, device=device) * 220 + 200

    def step(self, t_val, dt=0.05):
        flow, rgb = gpu_stream_flow(self.model, self.P, t_val,
                                    swirl=self.swirl, drift=self.drift)
        speed = flow.norm(dim=1, keepdim=True) + 1e-6
        flow = flow / torch.clamp(speed / 2.5, min=1.0)
        self.P = self.P + flow * dt
        self.age += 1
        out = ((self.P[:, 0] < -self.dom_w) | (self.P[:, 0] > self.dom_w) |
               (self.P[:, 1] >  self.dom_h) | (self.age > self.life))
        self._respawn(out)
        return flow, rgb

In [ ]:
# Inline animation — 900 frames at 22fps (~41s loop).
# dt slows particle motion; PERIOD slows the field's t-morph.
# Seeds: 1=red flames, 7=cyan/magenta, 23=warm/cold split,
# 42=green aurora, 99=teal smoke. Default is 42.
GPU_DOM_W, GPU_DOM_H = 1.5, 4.5
gpu_cppn  = CPPN(seed=42).to(device)
gpu_sim   = GPUNeuralFlow(gpu_cppn, n=60000, dom_w=GPU_DOM_W, dom_h=GPU_DOM_H,
                          swirl=1.5, drift=(0.0, 0.12), seed=0)
gpu_canvas = GPUTrailCanvas(w=200, h=600, fade=0.97)

FRAMES = 900
PERIOD = 900     # one full t-cycle per loop — slow, meditative

fig = plt.figure(figsize=(3.33, 10), dpi=90)
fig.patch.set_facecolor('#000')
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor('#000'); ax.axis('off')
img = ax.imshow(gpu_canvas.render(), origin='lower', interpolation='bilinear', aspect='auto')

def gpu_update(frame):
    t_val = 0.5 * np.sin(2 * np.pi * frame / PERIOD)
    flow, rgb = gpu_sim.step(t_val, dt=0.025)
    speed1d = torch.clamp(flow.norm(dim=1), max=3.0)
    intensity = 0.018 + 0.012 * speed1d
    gpu_canvas.splat(gpu_sim.P, intensity, rgb, gpu_sim.dom_w, gpu_sim.dom_h)
    img.set_data(gpu_canvas.render(gamma=0.7, scale=1.0))
    return (img,)

gpu_anim = animation.FuncAnimation(fig, gpu_update, frames=FRAMES, interval=45, blit=True)
plt.close(fig)
HTML(gpu_anim.to_jshtml(default_mode='loop'))

In [ ]:
# Save as MP4 (small, smooth) if ffmpeg is available, else fall back to GIF.
import shutil
if shutil.which('ffmpeg'):
    writer = animation.FFMpegWriter(fps=22, bitrate=2400,
                                    codec='libx264',
                                    extra_args=['-pix_fmt', 'yuv420p',
                                                '-preset', 'medium'])
    gpu_anim.save('neural_flow_gpu.mp4', writer=writer)
    print('Saved → neural_flow_gpu.mp4')
else:
    gpu_anim.save('neural_flow_gpu.gif', writer=animation.PillowWriter(fps=22))
    print('Saved → neural_flow_gpu.gif (install ffmpeg for a smaller MP4)')

In [ ]:
# A high-resolution still at the actual aspect of a 30"×10" display.
# 540×1620 → 1.6 MP, plenty for a desktop wallpaper or printed poster.
hi_cppn   = CPPN(seed=42).to(device)
hi_sim    = GPUNeuralFlow(hi_cppn, n=110000, dom_w=GPU_DOM_W, dom_h=GPU_DOM_H,
                          swirl=1.5, drift=(0.0, 0.2), seed=1)
hi_canvas = GPUTrailCanvas(w=540, h=1620, fade=0.965)

for f in range(800):
    t_val = 0.5 * np.sin(2 * np.pi * f / 500)
    flow, rgb = hi_sim.step(t_val, dt=0.04)
    speed1d = torch.clamp(flow.norm(dim=1), max=3.0)
    intensity = 0.015 + 0.01 * speed1d
    hi_canvas.splat(hi_sim.P, intensity, rgb, hi_sim.dom_w, hi_sim.dom_h)

fig, ax = plt.subplots(figsize=(3.33, 10), dpi=200)
fig.patch.set_facecolor('#000'); ax.axis('off')
ax.imshow(hi_canvas.render(gamma=0.7, scale=1.0), origin='lower', aspect='auto')
plt.savefig('neural_flow_gpu_still.png', facecolor='#000', dpi=200,
            bbox_inches='tight', pad_inches=0)
plt.show()
print('Saved → neural_flow_gpu_still.png')

## Knobs to turn

Every parameter below changes the screensaver in a noticeable way. Worth
spending five minutes just twiddling them.

| Knob | Where | Effect |
| --- | --- | --- |
| `seed` on `CPPN(seed=…)` | model construction | Completely new field. Try 1, 7, 17, 23, 42, 99. |
| `w_scale` (default 8.0) | `CPPN.__init__` | Higher = more chaotic detail; lower = smoother. |
| `n_layers` (default 4) | `CPPN.__init__` | Deeper = more compositional structure. |
| `n` (default 90k particles) | `GPUNeuralFlow` | More particles = denser glow. GPU can handle 500k. |
| `swirl` (default 1.5) | `GPUNeuralFlow` | How strongly particles follow the curl. |
| `drift` (default `(0, 0.2)`) | `GPUNeuralFlow` | Constant velocity; gives the "rising embers" feel. |
| `fade` (default 0.95) | `GPUTrailCanvas` | Closer to 1.0 = longer trails. |
| `PERIOD` (default 320) | render loop | Frames per t-cycle. Larger = slower morph. |
| `t` scale (default 0.5·sin) | `gpu_update` | Larger amplitude = the field morphs through more states per loop. |

## Where to take it next (GPU edition)

- **Replace `t = sin(2πf/P)` with a Lissajous curve in time**: `(sin t, cos 1.7t)`
  as a 2D time input. The field's loop never repeats but always returns near home.
- **Two CPPNs blended by α(t)**: morph between two random "personalities."
- **Train the CPPN on something.** Fit it to an image, an audio spectrogram, or
  noise from a tiny VAE. The flow inherits the structure of whatever you feed it.
- **Add neighbor interactions** (cheap on GPU): repulsion between nearby
  particles turns the flow into a Boids-like swarm.
- **Render frames to disk → encode with ffmpeg → loop as wallpaper.** On Linux,
  `xwinwrap` lets you set an animated wallpaper from any video file.

The GPU version is about 100 lines of PyTorch on top of the educational
NumPy code. Same math, more horsepower, more knobs.